# 1.프로젝트 개요

# 02. Data Preprocessing

## HR Analytics: 직원 이직 요인 및 조직 관리 분석

### 1. 전처리 목적

01_Data_Overview 단계에서 확인한 데이터 구조와 품질 점검 결과를 바탕으로 분석에 적합한 형태로 데이터를 정리한다.

본 단계에서는 불필요한 변수 검토, 데이터 타입 확인, 범주형 데이터 정리 및 파생변수 생성을 수행한다.

### 2. 주요 전처리 항목

- 결측치 및 중복 데이터 재확인
- 분석 목적에 맞지 않는 변수 검토
- 데이터 타입 및 범주값 검증
- 범주형 변수의 일관성 확보
- 분석 목적에 맞는 파생변수 생성
- 전처리 결과 검증 및 저장

### 3. 전처리 원칙

데이터의 정보 손실을 최소화하는 것을 원칙으로 한다.

통계적 기준에서 이상치로 분류된 값이라도 실제 관측 가능한 값이라면 임의로 제거하지 않는다.

또한 범주형 변수는 탐색적 분석 단계에서 원래의 의미를 유지하고, 모델링에 필요한 인코딩은 별도의 모델링 단계에서 수행한다.

# 라이브러리 및 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

file_path = "../data/raw/hr.csv"

raw_df = pd.read_csv(file_path)

hr_df = raw_df.copy()

print("원본 데이터 크기:", raw_df.shape)
print("전처리 작업 데이터 크기:", hr_df.shape)

원본 데이터 크기: (1470, 35)
전처리 작업 데이터 크기: (1470, 35)


# 3. 데이터 품질 재확인


## 3. 데이터 품질 재확인

전처리에 앞서 결측치, 중복 데이터 및 데이터 타입을 다시 확인한다.

이를 통해 데이터의 현재 상태를 파악하고 실제 처리가 필요한 항목을 선정한다.

불필요한 데이터 변환이나 삭제를 방지하기 위해 확인된 문제에 대해서만 전처리를 수행한다.


In [2]:
# 데이터 구조 확인
hr_df.info()

# 결측치 확인
missing_df = pd.DataFrame({
    "missing_count": hr_df.isnull().sum(),
    "missing_pct": (hr_df.isnull().mean() * 100).round(2)
})

display(missing_df)

# 중복 데이터 확인
print("완전 중복 행:", hr_df.duplicated().sum())
print(
    "EmployeeNumber 중복:",
    hr_df["EmployeeNumber"].duplicated().sum()
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

,missing_count,missing_pct
Age,0,0.0
Attrition,0,0.0
BusinessTravel,0,0.0
DailyRate,0,0.0
Department,0,0.0
DistanceFromHome,0,0.0
Education,0,0.0
EducationField,0,0.0
EmployeeCount,0,0.0
EmployeeNumber,0,0.0


완전 중복 행: 0
EmployeeNumber 중복: 0



## 4. 불필요한 변수 검토

분석에 활용할 변수와 분석 목적에 직접적인 의미가 없는 변수를 구분한다.

특히 모든 관측값에서 동일한 값을 가지는 상수형 변수는 분석에 추가적인 정보를 제공하지 않으므로 제거 대상으로 검토한다.

직원 식별번호는 개별 직원을 구분하기 위한 목적으로만 사용하며, 직원의 이직 특성을 설명하는 분석 변수로 사용하지 않는다.

### 변수 검토 기준

- EmployeeCount: 모든 행에서 동일한 값이므로 분석에서 제외
- Over18: 모든 행에서 동일한 값이므로 분석에서 제외
- StandardHours: 모든 행에서 동일한 값이므로 분석에서 제외
- EmployeeNumber: 직원 식별을 위한 ID이므로 설명변수에서 제외

단, EmployeeNumber는 데이터 추적 및 중복 검증을 위해 전처리 데이터에는 유지한다.


In [3]:
# 고유값이 1개인 상수형 변수 확인
constant_cols = [
    col for col in hr_df.columns
    if hr_df[col].nunique(dropna=False) == 1
]

print("상수형 변수:", constant_cols)

# 식별자 변수
id_cols = ["EmployeeNumber"]

# 분석에서 제외할 변수
drop_cols = constant_cols

# 상수형 변수 제거
hr_df = hr_df.drop(columns=drop_cols)

print("제거된 변수:", drop_cols)
print("현재 데이터 크기:", hr_df.shape)

상수형 변수: ['EmployeeCount', 'Over18', 'StandardHours']
제거된 변수: ['EmployeeCount', 'Over18', 'StandardHours']
현재 데이터 크기: (1470, 32)



## 5. 데이터 타입 및 값 검증

각 변수의 데이터 타입과 실제 값을 확인하여 분석 목적에 적합한 형태로 저장되어 있는지 검토한다.

특히 이직 여부, 초과근무 여부 등 주요 범주형 변수의 값이 예상된 범주로 구성되어 있는지 확인한다.

또한 나이, 급여, 근속연수 등 정량적 변수에서 비정상적인 음수 값이 존재하는지 점검한다.


In [4]:
# 주요 범주형 변수의 고유값 확인
check_cols = [
    "Attrition",
    "OverTime",
    "Department",
    "JobRole",
    "Gender",
    "BusinessTravel"
]

for col in check_cols:
    print(f"\n[{col}]")
    print(hr_df[col].value_counts(dropna=False))


[Attrition]
Attrition
No     1233
Yes     237
Name: count, dtype: int64

[OverTime]
OverTime
No     1054
Yes     416
Name: count, dtype: int64

[Department]
Department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64

[JobRole]
JobRole
Sales Executive              326
Research Scientist           292
Laboratory Technician        259
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director             80
Human Resources               52
Name: count, dtype: int64

[Gender]
Gender
Male      882
Female    588
Name: count, dtype: int64

[BusinessTravel]
BusinessTravel
Travel_Rarely        1043
Travel_Frequently     277
Non-Travel            150
Name: count, dtype: int64


In [5]:
# 정량적 변수의 음수 여부 확인
numeric_check_cols = [
    "Age",
    "MonthlyIncome",
    "DistanceFromHome",
    "YearsAtCompany",
    "YearsInCurrentRole",
    "YearsWithCurrManager"
]

negative_check = (
    hr_df[numeric_check_cols] < 0
).sum()

display(negative_check.to_frame("negative_count"))

,negative_count
Age,0
MonthlyIncome,0
DistanceFromHome,0
YearsAtCompany,0
YearsInCurrentRole,0
YearsWithCurrManager,0



### 5-1. 데이터 타입 및 범주형 변수 처리 기준

데이터 타입만으로 변수의 통계적 특성을 판단하지 않도록 주의한다.

- 명목형 변수: Department, JobRole, Gender 등
- 이진형 변수: Attrition, OverTime 등
- 순서형 변수: JobSatisfaction, EnvironmentSatisfaction, WorkLifeBalance 등
- 정량형 변수: Age, MonthlyIncome, YearsAtCompany 등
- 식별자 변수: EmployeeNumber

순서형 변수는 숫자로 저장되어 있지만 각 숫자는 등급을 의미하므로 일반적인 연속형 수치 변수와 구분하여 해석한다.



## 6. 범주형 변수 정리

범주형 변수의 앞뒤 공백과 표기 차이를 점검하고, 동일한 의미를 가진 범주가 일관되게 관리되도록 정리한다.

다만 원본 데이터에 존재하는 범주 자체를 임의로 통합하거나 변경하지 않는다.


In [7]:
categorical_cols = hr_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# 문자열 앞뒤 공백 제거
for col in categorical_cols:
    hr_df[col] = hr_df[col].str.strip()

# 정리 후 범주형 변수 확인
for col in categorical_cols:
    print(f"{col}: {hr_df[col].nunique()}개")

Attrition: 2개
BusinessTravel: 3개
Department: 3개
EducationField: 6개
Gender: 2개
JobRole: 9개
MaritalStatus: 3개
OverTime: 2개



## 7. 파생변수 생성

기존 변수만으로 파악하기 어려운 직원의 특성을 추가적으로 분석하기 위해 파생변수를 생성한다.

본 단계에서는 근속연수를 기준으로 직원의 근속 구간을 구분한다.

이를 통해 재직 기간에 따른 직원 집단별 이직률을 비교할 수 있도록 분석 기반을 마련한다.

파생변수는 원본 데이터를 유지하면서 새로운 분석 관점을 제공하는 목적으로 활용한다.


In [9]:
# 근속연수 구간 생성
hr_df["TenureGroup"] = pd.cut(
    hr_df["YearsAtCompany"],
    bins=[-1, 1, 3, 5, 10, np.inf],
    labels=[
        "0-1 Years",
        "2-3 Years",
        "4-5 Years",
        "6-10 Years",
        "Over 10 Years"
    ]
)

display(
    hr_df[["YearsAtCompany", "TenureGroup"]].head(10)
)

,YearsAtCompany,TenureGroup
0,6,6-10 Years
1,10,6-10 Years
2,0,0-1 Years
3,8,6-10 Years
4,2,2-3 Years
5,7,6-10 Years
6,1,0-1 Years
7,1,0-1 Years
8,9,6-10 Years
9,7,6-10 Years



### 7-1. 이직 여부 수치형 변수 생성

이직 여부를 나타내는 Attrition 변수는 Yes와 No로 구성되어 있다.

향후 통계 분석 및 머신러닝 모델링에서 활용할 수 있도록 이직 여부를 1과 0으로 변환한 파생변수를 생성한다.

- Yes: 1
- No: 0

기존 Attrition 변수는 집단별 분석과 시각화에 활용하기 위해 유지한다.


In [10]:
hr_df["AttritionBinary"] = (
    hr_df["Attrition"].map({
        "Yes": 1,
        "No": 0
    })
)

display(
    hr_df[["Attrition", "AttritionBinary"]]
    .drop_duplicates()
)

,Attrition,AttritionBinary
0,Yes,1
1,No,0



## 8. 최종 데이터 검증

전처리가 완료된 데이터의 구조와 품질을 다시 확인한다.

전처리 과정에서 데이터 행이 불필요하게 삭제되거나 새로운 결측치가 발생하지 않았는지 검토한다.

또한 파생변수가 정상적으로 생성되었는지 확인하여 다음 분석 단계에서 활용할 수 있는 상태인지 평가한다.


In [11]:
# 전처리 전후 데이터 비교
comparison_df = pd.DataFrame({
    "구분": ["원본 데이터", "전처리 데이터"],
    "행 개수": [raw_df.shape[0], hr_df.shape[0]],
    "열 개수": [raw_df.shape[1], hr_df.shape[1]],
    "결측치": [
        raw_df.isnull().sum().sum(),
        hr_df.isnull().sum().sum()
    ],
    "중복 행": [
        raw_df.duplicated().sum(),
        hr_df.duplicated().sum()
    ]
})

display(comparison_df)

,구분,행 개수,열 개수,결측치,중복 행
0,원본 데이터,1470,35,0,0
1,전처리 데이터,1470,34,0,0


In [12]:
# 파생변수 검증
print("이직 여부 매핑 결과")
display(hr_df["AttritionBinary"].value_counts(dropna=False))

print("\n근속 구간별 데이터 개수")
display(hr_df["TenureGroup"].value_counts(dropna=False))

# 최종 데이터 정보
hr_df.info()

이직 여부 매핑 결과


AttritionBinary
0    1233
1     237
Name: count, dtype: int64


근속 구간별 데이터 개수


TenureGroup
6-10 Years       448
4-5 Years        306
2-3 Years        255
Over 10 Years    246
0-1 Years        215
Name: count, dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Age                       1470 non-null   int64   
 1   Attrition                 1470 non-null   object  
 2   BusinessTravel            1470 non-null   object  
 3   DailyRate                 1470 non-null   int64   
 4   Department                1470 non-null   object  
 5   DistanceFromHome          1470 non-null   int64   
 6   Education                 1470 non-null   int64   
 7   EducationField            1470 non-null   object  
 8   EmployeeNumber            1470 non-null   int64   
 9   EnvironmentSatisfaction   1470 non-null   int64   
 10  Gender                    1470 non-null   object  
 11  HourlyRate                1470 non-null   int64   
 12  JobInvolvement            1470 non-null   int64   
 13  JobLevel                  1470 non-null   int64 


## 9. 전처리 결과 저장

전처리 및 검증이 완료된 데이터를 CSV 파일로 저장한다.

저장된 데이터는 이후 탐색적 데이터 분석, 통계적 가설 검정 및 머신러닝 분석에 활용한다.

원본 데이터는 별도로 보존하여 추후 전처리 과정의 재현성과 데이터 비교가 가능하도록 한다.


In [13]:
import os

save_path = "../data/processed"

os.makedirs(save_path, exist_ok=True)

hr_df.to_csv(
    os.path.join(save_path, "hr_preprocessed.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("전처리 데이터 저장 완료")
print("저장 경로:", save_path)

전처리 데이터 저장 완료
저장 경로: ../data/processed



## 10. 전처리 결과 및 다음 단계

### 전처리 수행 내용

- 데이터의 결측치 및 중복 여부 재확인
- 분석에 불필요한 상수형 변수 제거
- 직원 식별번호 유지 및 분석 변수와 구분
- 데이터 타입과 범주형 변수 검증
- 근속연수 구간 파생변수 생성
- 이직 여부 수치형 파생변수 생성
- 전처리 데이터 저장

### 다음 단계

전처리된 데이터를 활용하여 직원의 인구통계학적 특성, 직무, 급여, 근속연수 및 근무 환경에 따른 이직 현황을 탐색한다.

다음 단계에서는 직원 집단별 이직률을 비교하고 주요 변수 간의 관계를 시각적으로 분석한다.
